# 🚀 Databricks Inventory Reconciliation Engine (AWS S3 & Glue)
### Enterprise Time-Series Inventory Health & Depletion Analysis across Full Horizon

This Databricks notebook executes the high-performance daily time-series inventory reconciliation pipeline natively on **Databricks Spark Runtime** reading from and writing to **AWS S3** and **AWS Glue Data Catalog**.

---

### 🏛️ Architecture of the inventory Reconciliation Job:
- **Compute Engine**: Databricks Spark Cluster (Runtime 16.x)
- **Source Lakehouse (AWS S3)**: `s3://cymbal-global-retail-demo/lakehouse/elevate_data/`
- **AWS Glue Database**: `latest_elevate_data` (with fallback to `elevate_data`, `aws_lakehouse`, `omni_glue_db`)
- **Target Sink (AWS S3 Bucket)**: `s3://cymbal-global-retail-demo/lakehouse/gold_inventory_reconciliation_ledger`
- **Target Catalog Table**: `aws_glue_federated_catalog.gold_inventory_reconciliation_ledger`

In [0]:
# Databricks Widget Parameters for Job Scheduling & Interactive Runs
import os
import sys

try:
    # Reset stale cached widgets from earlier interactive sessions
    dbutils.widgets.removeAll()
    
    # Register fresh widget parameters
    dbutils.widgets.text("aws_access_key", "", "AWS Access Key")
    dbutils.widgets.text("aws_secret_key", "", "AWS Secret Access Key")
    dbutils.widgets.text("aws_glue_catalog_id", "621785110540", "AWS Glue Catalog ID")
    dbutils.widgets.text("aws_region", "us-east-1", "AWS Region")
    dbutils.widgets.text("database", "latest_elevate_data", "Glue Database Name")
    dbutils.widgets.text("s3_lakehouse_base", "s3://rakeshmohandas-uscentral1-621785110540-us-east-1-an/lakehouse/latest_elevate_data", "S3 Lakehouse Source Base")
    dbutils.widgets.text("s3_sink_path", "s3://rakeshmohandas-uscentral1-621785110540-us-east-1-an/lakehouse/latest_elevate_data/gold_inventory_reconciliation_ledger", "S3 Target Sink Path")
    dbutils.widgets.text("target_table", "latest_elevate_data.gold_inventory_reconciliation_ledger", "Target Output Table")
    dbutils.widgets.text("start_date", "2025-01-01", "Evaluation Start Date")
    dbutils.widgets.text("until_date", "2026-09-10", "Evaluation Until Date")
    dbutils.widgets.text("write_mode", "overwrite", "Write Mode (overwrite/append)")
    dbutils.widgets.text("dry_run", "False", "Dry Run Mode (True/False)")
    dbutils.widgets.text("critical_cover_hours", "6.0", "Critical Cover Hours Threshold")
    dbutils.widgets.text("monitor_cover_hours", "12.0", "Monitor Cover Hours Threshold")
    dbutils.widgets.text("dbu_rate_per_hour", "1.50", "Databricks Cluster DBU Rate/hr")
    dbutils.widgets.text("dbu_dollar_rate", "0.40", "Cost per DBU ($ USD)")
    
    AWS_ACCESS_KEY       = dbutils.widgets.get("aws_access_key")
    AWS_SECRET_KEY       = dbutils.widgets.get("aws_secret_key")
    AWS_GLUE_CATALOG     = dbutils.widgets.get("aws_glue_catalog_id")
    AWS_REGION           = dbutils.widgets.get("aws_region")
    DATABASE             = dbutils.widgets.get("database")
    S3_LAKEHOUSE_BASE    = dbutils.widgets.get("s3_lakehouse_base")
    S3_SINK_PATH         = dbutils.widgets.get("s3_sink_path")
    TARGET_TABLE         = dbutils.widgets.get("target_table")
    START_DATE           = dbutils.widgets.get("start_date")
    UNTIL_DATE           = dbutils.widgets.get("until_date")
    WRITE_MODE           = dbutils.widgets.get("write_mode")
    DRY_RUN              = dbutils.widgets.get("dry_run").lower() in ["true", "1", "yes"]
    CRITICAL_THRESHOLD   = float(dbutils.widgets.get("critical_cover_hours"))
    MONITOR_THRESHOLD    = float(dbutils.widgets.get("monitor_cover_hours"))
    DBU_RATE_PER_HOUR    = float(dbutils.widgets.get("dbu_rate_per_hour"))
    DBU_DOLLAR_RATE      = float(dbutils.widgets.get("dbu_dollar_rate"))
except NameError:
    # Fallback for standalone / non-Databricks execution
    AWS_ACCESS_KEY       = os.getenv("AWS_ACCESS_KEY_ID", "XXXXXXXXXXXXXXXXXXXX")
    AWS_SECRET_KEY       = os.getenv("AWS_SECRET_ACCESS_KEY", "6f9vzbYPRTbYYe3iskQwMFbMEyaP08US6UQils6n")
    AWS_GLUE_CATALOG     = os.getenv("AWS_GLUE_CATALOG_ID", "621785110540")
    AWS_REGION           = os.getenv("AWS_REGION", "us-east-1")
    DATABASE             = os.getenv("DATABASE", "latest_elevate_data")
    S3_LAKEHOUSE_BASE    = os.getenv("S3_LAKEHOUSE_BASE", "s3://rakeshmohandas-uscentral1-621785110540-us-east-1-an/lakehouse/latest_elevate_data")
    S3_SINK_PATH         = os.getenv("S3_SINK_PATH", "s3://rakeshmohandas-uscentral1-621785110540-us-east-1-an/lakehouse/latest_elevate_data/gold_inventory_reconciliation_ledger")
    TARGET_TABLE         = os.getenv("TARGET_TABLE", "latest_elevate_data.gold_inventory_reconciliation_ledger")
    START_DATE           = os.getenv("START_DATE", "2025-01-01")
    UNTIL_DATE           = os.getenv("UNTIL_DATE", "2026-09-10")
    WRITE_MODE           = os.getenv("WRITE_MODE", "overwrite")
    DRY_RUN              = os.getenv("DRY_RUN", "False").lower() in ["true", "1", "yes"]
    CRITICAL_THRESHOLD   = float(os.getenv("CRITICAL_THRESHOLD", "6.0"))
    MONITOR_THRESHOLD    = float(os.getenv("MONITOR_THRESHOLD", "12.0"))
    DBU_RATE_PER_HOUR    = float(os.getenv("DBU_RATE_PER_HOUR", "1.50"))
    DBU_DOLLAR_RATE      = float(os.getenv("DBU_DOLLAR_RATE", "0.40"))

# Configure AWS S3 Credentials in Spark Session
if AWS_ACCESS_KEY and AWS_SECRET_KEY:
    for prefix in ["fs.s3a", "spark.hadoop.fs.s3a"]:
        try:
            spark.conf.set(f"{prefix}.access.key", AWS_ACCESS_KEY)
            spark.conf.set(f"{prefix}.secret.key", AWS_SECRET_KEY)
            spark.conf.set(f"{prefix}.endpoint", f"s3.{AWS_REGION}.amazonaws.com")
            spark.conf.set(f"{prefix}.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
            spark.conf.set(f"{prefix}.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
        except Exception:
            pass

print("=" * 100)
print("✅ Databricks Spark AWS S3 & Glue Configuration Initialized")
print(f"   - AWS Glue Catalog ID  : {AWS_GLUE_CATALOG}")
print(f"   - AWS Region           : {AWS_REGION}")
print(f"   - Source Database      : {DATABASE}")
print(f"   - S3 Lakehouse Source  : {S3_LAKEHOUSE_BASE}")
print(f"   - S3 Target Sink       : {S3_SINK_PATH}")
print(f"   - Target Table (Glue)  : {TARGET_TABLE}")
print(f"   - Evaluation Horizon   : {START_DATE} to {UNTIL_DATE}")
print(f"   - Dry Run Mode         : {DRY_RUN}")
print(f"   - DBU Unit Rate        : {DBU_RATE_PER_HOUR:.2f} DBU/hr (@ ${DBU_DOLLAR_RATE:.2f}/DBU)")
print("=" * 100)

✅ Databricks Spark AWS S3 & Glue Configuration Initialized
   - AWS Glue Catalog ID  : 825834484882
   - AWS Region           : us-east-1
   - Source Database      : latest_elevate_data
   - S3 Lakehouse Source  : s3://rakeshmohandas-uscentral1-825834484882-us-east-1-an/lakehouse/latest_elevate_data
   - S3 Target Sink       : s3://rakeshmohandas-uscentral1-825834484882-us-east-1-an/lakehouse/latest_elevate_data/gold_inventory_reconciliation_ledger
   - Target Table (Glue)  : latest_elevate_data.gold_inventory_reconciliation_ledger
   - Evaluation Horizon   : 2025-01-01 to 2026-09-10
   - Dry Run Mode         : False
   - DBU Unit Rate        : 1.50 DBU/hr (@ $0.40/DBU)


In [0]:
import logging
import sys
import time
import os
from datetime import datetime, timezone
from typing import List, Optional, Tuple

from pyspark.sql import SparkSession, DataFrame, Window
from pyspark.sql import functions as F
from pyspark.sql.types import (
    ArrayType, StructType, StructField, StringType, IntegerType, DoubleType
)

# Schema Constants - Conformed 12-Column Contract
LEDGER_COLUMNS = [
    "business_date", "store_id", "store_name", "city", "item_id", "unit_price_usd",
    "opening_qty", "shelf_qty", "backroom_qty", "intraday_gross_revenue_usd",
    "est_cover_hours_remaining", "reconciliation_status",
]

STATUS_CRITICAL = "CRITICAL BURN SPIKE - STOCKOUT RISK"
STATUS_MONITOR  = "MONITOR VELOCITY"
STATUS_NORMAL   = "RECONCILED NORMAL HEALTH"

SAFETY_STOCK_UNITS   = 3
PEAK_HOUR            = 17
CITY_CANDIDATES      = ["city", "store_city", "locality", "town", "municipality"]
CITY_FALLBACKS       = ["country", "region", "global_region", "market"]

logging.basicConfig(
    level=logging.INFO,
    format="[%(asctime)s] [%(levelname)-7s] [%(name)-24s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
)
log = logging.getLogger("DatabricksReconciliation")


class ContractError(RuntimeError):
    """Raised when ledger schema drifts from contract or row count does not match position grid."""


class DatabricksReconciliationPipeline:
    """
    Databricks Multi-Day Inventory Reconciliation Engine
    ====================================================
    - Ingests POS stream, store inventory baseline, and store dimensions directly from AWS S3 Lakehouse.
    - Evaluates 5-minute intraday depletion curves shaped around 5:00 PM peak rush.
    - Sinks conformed 12-column ledger directly to AWS S3 & registers table in AWS Glue Catalog.
    - Exact 1-to-1 logic parity with Dataproc Serverless engine.
    """
    def __init__(
        self,
        spark_session: SparkSession,
        database: str = "latest_elevate_data",
        s3_lakehouse_base: str = "s3://rakeshmohandas-uscentral1-825834484882-us-east-1-an/lakehouse/latest_elevate_data",
        s3_sink_path: str = "s3://rakeshmohandas-uscentral1-825834484882-us-east-1-an/lakehouse/latest_elevate_data/gold_inventory_reconciliation_ledger",
        target_table: str = "latest_elevate_data.gold_inventory_reconciliation_ledger",
        start_date: str = "2025-01-01",
        until_date: str = "2026-09-10",
        horizon_days: int = 1,
        slot_minutes: int = 5,
        critical_cover_hours: float = 6.0,
        monitor_cover_hours: float = 12.0,
        expr_passes: int = 1,
        write_mode: str = "overwrite",
        dry_run: bool = False
    ):
        self.spark = spark_session
        self.database = database
        self.s3_lakehouse_base = s3_lakehouse_base.rstrip("/")
        self.s3_sink_path = s3_sink_path.rstrip("/")
        self.target_table = target_table
        self.start_date = start_date
        self.until_date = until_date
        self.horizon_days = horizon_days
        self.slot_minutes = slot_minutes
        self.critical_cover_hours = critical_cover_hours
        self.monitor_cover_hours = monitor_cover_hours
        self.expr_passes = expr_passes
        self.write_mode = write_mode
        self.dry_run = dry_run

        if 60 % slot_minutes:
            raise ValueError(f"slot_minutes must divide 60, got {slot_minutes}")
        self.slots_per_hour = 60 // slot_minutes
        self.slots_per_day  = 24 * self.slots_per_hour
        self.total_slots    = horizon_days * self.slots_per_day
        self.hours_per_slot = slot_minutes / 60.0
        self.n_positions: Optional[int] = None

    @staticmethod
    def _pick(df: DataFrame, names: List[str]) -> Optional[str]:
        low = {c.lower(): c for c in df.columns}
        return next((low[n] for n in names if n in low), None)

    # 1. READ FROM AWS S3 LAKEHOUSE DIRECTLY
    def read_sources(self) -> Tuple[DataFrame, DataFrame, DataFrame]:
        log.info("[1/7 READ] Ingesting date range '%s' to '%s' from AWS S3 Lakehouse: %s", self.start_date, self.until_date, self.s3_lakehouse_base)
        pos_where = ""
        if self.start_date and self.until_date:
            pos_where = f" WHERE business_date >= '{self.start_date}' AND business_date <= '{self.until_date}'"
        elif self.until_date and self.until_date.lower() != "all":
            pos_where = f" WHERE business_date <= '{self.until_date}'"

        def rd(table_name: str, where_clause: str = "") -> DataFrame:
            # 1. Primary: Direct AWS S3 Storage Parquet Paths (exact parity with Dataproc)
            s3_bases = [
                self.s3_lakehouse_base,
                self.s3_lakehouse_base.replace("s3://", "s3a://"),
                "s3://rakeshmohandas-uscentral1-825834484882-us-east-1-an/lakehouse/latest_elevate_data",
                "s3a://rakeshmohandas-uscentral1-825834484882-us-east-1-an/lakehouse/latest_elevate_data"
            ]
            for base in s3_bases:
                paths = [
                    f"{base}/{table_name}/data/*.parquet",
                    f"{base}/{table_name}.parquet",
                    f"{base}/{table_name}/*.parquet",
                    f"{base}/{table_name}"
                ]
                for p in paths:
                    try:
                        df_raw = self.spark.read.parquet(p)
                        df_raw.createOrReplaceTempView(f"tmp_{table_name}")
                        df = self.spark.sql(f"SELECT * FROM tmp_{table_name}{where_clause}")
                        log.info("  ✅ Loaded table '%s' directly from AWS S3: `%s`", table_name, p)
                        return df
                    except Exception:
                        continue

            # 2. Secondary fallback: Glue Catalog database for latest_elevate_data only
            glue_candidates = [
                f"latest_elevate_data.{table_name}",
                f"{self.database}.{table_name}",
                f"workspace.{self.database}.{table_name}",
            ]
            for cand in glue_candidates:
                try:
                    df = self.spark.sql(f"SELECT * FROM {cand}{where_clause}")
                    log.info("  ✅ Loaded table '%s' via Glue table: `%s`", table_name, cand)
                    return df
                except Exception:
                    continue

            raise RuntimeError(f"Could not load table '{table_name}' from AWS S3 Lakehouse: {self.s3_lakehouse_base}/{table_name}")

        bronze = rd("bronze_pos_stream_events", pos_where)
        inv    = rd("silver_store_inventory")
        nodes  = rd("store_nodes")
        return bronze, inv, nodes

    # 2. SILVER POS TRANSFORMATION
    def silver_pos(self, bronze: DataFrame) -> Tuple[DataFrame, DataFrame, DataFrame]:
        log.info("[2/7 SILVER POS] Processing POS transactions for range: %s to %s", self.start_date, self.until_date)
        have = {c.lower() for c in bronze.columns}
        order = [F.col(c).desc_nulls_last() for c in ("_ingested_at", "_msk_message_id") if c in have] or [F.lit(1)]

        # Handle items JSON string if not already parsed as array
        if "items" in bronze.columns and str(bronze.schema["items"].dataType).lower().startswith("string"):
            item_schema = ArrayType(StructType([
                StructField("line_seq", IntegerType(), True),
                StructField("item_id", StringType(), True),
                StructField("item_name", StringType(), True),
                StructField("category", StringType(), True),
                StructField("quantity", IntegerType(), True),
                StructField("unit_price", DoubleType(), True),
                StructField("unit_price_usd", DoubleType(), True),
                StructField("total_item_price", DoubleType(), True),
                StructField("item_discount", DoubleType(), True),
                StructField("item_net_amount", DoubleType(), True)
            ]))
            bronze = bronze.withColumn("items", F.from_json(F.col("items"), item_schema))

        txn = (bronze
               .withColumn("_rk", F.row_number().over(Window.partitionBy("transaction_id").orderBy(*order)))
               .filter(F.col("_rk") == 1)
               .withColumn("event_ts", F.coalesce(F.to_timestamp("event_timestamp", "yyyy-MM-dd HH:mm:ss"), F.to_timestamp("event_timestamp")))
               .withColumn("business_dt", F.coalesce(F.to_date("business_date", "yyyy-MM-dd"), F.to_date(F.col("event_ts"))))
               .filter(F.col("transaction_id").isNotNull() & F.col("store_id").isNotNull()))

        if self.start_date:
            txn = txn.filter(F.col("business_dt") >= F.to_date(F.lit(self.start_date)))
        if self.until_date and self.until_date.lower() != "all":
            txn = txn.filter(F.col("business_dt") <= F.to_date(F.lit(self.until_date)))

        dates_df = txn.select(F.col("business_dt").alias("business_date")).distinct()

        daily_store_revenue = txn.groupBy("business_dt", "store_id").agg(
            F.round(F.sum(F.coalesce(F.col("total_amount_usd").cast("double"), F.lit(0.0))), 2).alias("intraday_gross_revenue_usd")
        ).withColumnRenamed("business_dt", "business_date")

        lines = (txn.select("business_dt", "store_id", F.explode_outer("items").alias("ln"))
                    .select(F.col("business_dt").alias("business_date"),
                            F.col("store_id"),
                            F.col("ln.item_id").alias("item_id"),
                            F.col("ln.quantity").cast("double").alias("qty"))
                    .filter(F.col("item_id").isNotNull()))

        daily_sku_units = (lines.groupBy("business_date", "store_id", "item_id")
                                .agg(F.greatest(F.sum("qty"), F.lit(0.0)).alias("daily_units")))

        return dates_df, daily_store_revenue, daily_sku_units

    # 3. POSITION GRID
    def positions(self, inv: DataFrame, nodes: DataFrame,
                  dates_df: DataFrame, daily_store_revenue: DataFrame,
                  daily_sku_units: DataFrame) -> DataFrame:
        log.info("[3/7 POSITIONS] Building Grid: (dates x store_id x item_id)")
        city_col = self._pick(nodes, CITY_CANDIDATES) or self._pick(nodes, CITY_FALLBACKS)
        city_expr = F.col(city_col) if city_col else F.lit(None)
        dim = (nodes.select(F.col("store_id"),
                            F.col("store_name").cast("string").alias("store_name"),
                            city_expr.cast("string").alias("city"))
                    .dropDuplicates(["store_id"]))

        # Deduplicate base inventory to 1 baseline snapshot per store and item
        inv_baseline = inv.dropDuplicates(["store_id", "item_id"])

        base_inv = (inv_baseline.drop("store_name", "city")
                       .join(F.broadcast(dim), "store_id", "inner")
                       .withColumn("on_hand", (F.coalesce(F.col("shelf_qty"), F.lit(0)) + F.coalesce(F.col("backroom_qty"), F.lit(0))).cast("double")))

        grid = dates_df.crossJoin(F.broadcast(base_inv))

        pos = (grid.join(daily_store_revenue, ["business_date", "store_id"], "left")
                   .join(daily_sku_units, ["business_date", "store_id", "item_id"], "left")
                   .withColumn("intraday_gross_revenue_usd", F.coalesce(F.col("intraday_gross_revenue_usd"), F.lit(0.0)).cast("double"))
                   .withColumn("daily_units", F.coalesce(F.col("daily_units"), F.lit(0.0)).cast("double"))
                   .withColumn("pos_key", F.concat_ws("~", F.col("business_date").cast("string"), F.col("store_id"), F.col("item_id"))))

        self.n_positions = pos.count()
        log.info("  Grid: %s positions across evaluated horizon.", f"{self.n_positions:,}")
        return pos

    # 4. TIME FABRIC EXPANSION
    def fabric(self, pos: DataFrame) -> DataFrame:
        n = self.n_positions * self.total_slots
        log.info("[4/7 FABRIC] %s positions x %s slots = %s calculation rows",
                 f"{self.n_positions:,}", f"{self.total_slots:,}", f"{n:,}")
        slots = self.spark.range(0, self.total_slots).withColumnRenamed("id", "slot")
        return (pos.crossJoin(F.broadcast(slots))
                   .withColumn("slot", F.col("slot").cast("int"))
                   .withColumn("hour", F.pmod((F.col("slot") / F.lit(self.slots_per_hour)).cast("int"), F.lit(24))))

    # 5. DEMAND SHAPING & AUDIT HASH
    def project(self, fab: DataFrame) -> DataFrame:
        log.info("[5/7 PROJECT] Vectorized 5:00 PM Peak Demand Shaping & Cryptographic Auditing")
        phase = (F.col("hour") - F.lit(float(PEAK_HOUR))) * F.lit(3.141592653589793 / 12.0)
        shape = (F.pow(F.cos(phase), F.lit(4.0)) * F.lit(1.85) + F.lit(0.15)) / F.lit(0.84375)

        df = fab.withColumn("slot_demand", F.col("daily_units") * shape / F.lit(float(self.slots_per_day)))
        chain = F.concat_ws("|", F.col("pos_key"), F.col("slot").cast("string"))
        for i in range(self.expr_passes):
            chain = F.sha2(F.concat_ws("#", chain, F.lit(f"s{i}")), 256)

        return df.withColumn("audit_hash", chain)

    # 6. CUMULATIVE DEPLETION WINDOW
    def window(self, df: DataFrame) -> DataFrame:
        log.info("[6/7 WINDOW] Cumulative Depletion Window (Safety Stock <= %d units)", SAFETY_STOCK_UNITS)
        w_pos = (Window.partitionBy("pos_key").orderBy("slot")
                       .rowsBetween(Window.unboundedPreceding, Window.currentRow))
        return (df.withColumn("burned", F.sum("slot_demand").over(w_pos))
                  .withColumn("projected_on_hand", F.greatest(F.lit(0.0), F.col("on_hand") - F.col("burned")))
                  .withColumn("is_out", (F.col("projected_on_hand") <= F.lit(float(SAFETY_STOCK_UNITS))).cast("int")))

    # 7. DAILY GRAIN SYNTHESIS
    def collapse(self, df: DataFrame) -> DataFrame:
        log.info("[7/7 COLLAPSE] Synthesizing Daily Reconciliation Ledger")
        carry = ["business_date", "store_id", "store_name", "city", "item_id", "unit_price_usd",
                 "opening_qty", "shelf_qty", "backroom_qty", "intraday_gross_revenue_usd",
                 "on_hand", "daily_units"]

        return (df.groupBy("pos_key")
                  .agg(*[F.first(c, ignorenulls=True).alias(c) for c in carry],
                       F.min(F.when(F.col("is_out") == 1, F.col("slot"))).alias("out_slot"))
                  .withColumn("business_date", F.to_date(F.col("business_date")))
                  .withColumn("hourly_burn", F.col("daily_units") / F.lit(24.0))
                  .withColumn("est_cover_hours_remaining",
                              F.round(F.coalesce(
                                  F.col("out_slot") * F.lit(self.hours_per_slot),
                                  F.when(F.col("hourly_burn") > 0, F.col("on_hand") / F.col("hourly_burn"))), 1).cast("double"))
                  .withColumn("reconciliation_status",
                              F.when(F.col("est_cover_hours_remaining") <= F.lit(self.critical_cover_hours), F.lit(STATUS_CRITICAL))
                               .when(F.col("est_cover_hours_remaining") <= F.lit(self.monitor_cover_hours), F.lit(STATUS_MONITOR))
                               .otherwise(F.lit(STATUS_NORMAL)))
                  .select(*LEDGER_COLUMNS)
                  .orderBy(F.col("business_date").desc(), F.col("est_cover_hours_remaining").asc_nulls_last()))

    # REPORTING & CONTRACT VALIDATION
    def report(self, led: DataFrame) -> int:
        n = led.count()
        if list(led.columns) != LEDGER_COLUMNS:
            raise ContractError(f"Schema drift detected: expected {LEDGER_COLUMNS}, got {list(led.columns)}")
        if self.n_positions is not None and n != self.n_positions:
            raise ContractError(f"Row drift detected: {n:,} rows out vs {self.n_positions:,} positions in")
        log.info("✅ Contract Verified: 12 conformed columns, %s positions in -> %s rows out.", f"{self.n_positions:,}", f"{n:,}")

        print("\n" + "=" * 128)
        print(f"FULL HISTORICAL INVENTORY RECONCILIATION LEDGER ($ USD) [{self.start_date} to {self.until_date}]")
        print(f"Execution Timestamp: {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')}")
        print("=" * 128)
        
        cols = ["business_date", "store_id", "city", "item_id", "unit_price_usd", "shelf_qty", "est_cover_hours_remaining", "reconciliation_status"]
        led.select(*cols).show(15, truncate=False)
        
        print("-" * 128)
        print(f"Reconciliation Status Distribution for Full Horizon ({self.start_date} to {self.until_date}):")
        led.groupBy("reconciliation_status").count().orderBy(F.col("count").desc()).show(truncate=False)
        print("=" * 128 + "\n")
        return n

    # WRITE TO AWS S3 BUCKET & AWS GLUE / DELTA TABLE
    def write(self, led: DataFrame) -> None:
        if self.dry_run:
            log.info("[DRY-RUN] Dry run enabled. Skipping push to AWS S3 & Glue table.")
            return

        log.info("Committing reconciled ledger directly to AWS S3 Lakehouse: %s (mode=%s)...", self.s3_sink_path, self.write_mode)
        try:
            # 1. Write Parquet format directly to AWS S3
            (led.write
                .mode(self.write_mode)
                .parquet(self.s3_sink_path))
            log.info("  ✅ Committed Parquet format table to AWS S3: %s", self.s3_sink_path)
            
            # 2. Also register / save to Glue Catalog / Metastore table
            try:
                (led.write
                    .mode(self.write_mode)
                    .saveAsTable(self.target_table))
                log.info("  ✅ Successfully registered/updated AWS Glue table: %s", self.target_table)
            except Exception as ge:
                log.info("  ℹ️ Catalog saveAsTable note: %s. Direct S3 write is already complete.", ge)

        except Exception as e:
            log.error("❌ Failed write to AWS S3: %s", e)
            raise e

    # RUN PIPELINE
    def run(self) -> DataFrame:
        t0 = time.time()
        print("=" * 96)
        print(f"🚀 Starting Databricks Multi-Day Inventory Reconciliation Pipeline | {self.start_date} -> {self.until_date}")
        print(f"📍 AWS S3 Lakehouse Source Base : {self.s3_lakehouse_base}")
        print(f"🏛️ AWS Glue Catalog Database    : {self.database}")
        print(f"☁️ AWS S3 Target Sink           : {self.s3_sink_path}")
        print(f"📋 Target Glue Table            : {self.target_table}")
        print(f"⚡ Cover Thresholds             : Critical <= {self.critical_cover_hours:.1f}h | Monitor <= {self.monitor_cover_hours:.1f}h")
        print("=" * 96)

        bronze, inv, nodes = self.read_sources()
        dates_df, store_revenue, sku_units = self.silver_pos(bronze)
        pos = self.positions(inv, nodes, dates_df, store_revenue, sku_units)
        ledger = self.collapse(self.window(self.project(self.fabric(pos))))

        self.report(ledger)
        self.write(ledger)
        duration = time.time() - t0
        print(f"\n✅ Pipeline core stages completed successfully in {duration:.2f} seconds.")
        return ledger

print("✅ DatabricksReconciliationPipeline class loaded successfully.")

✅ DatabricksReconciliationPipeline class loaded successfully.


In [0]:
# Run the Inventory Reconciliation Pipeline & Compute Databricks Benchmark Metrics
job_start_time = time.time()
start_dt_str = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")

print("=" * 100)
print(f"🚀 EXECUTING DATABRICKS RECONCILIATION ENGINE ({START_DATE} to {UNTIL_DATE})")
print(f"🏛️ AWS Glue Catalog Database : {DATABASE}")
print(f"☁️ AWS S3 Lakehouse Source   : {S3_LAKEHOUSE_BASE}")
print(f"☁️ AWS S3 Target Sink Bucket : {S3_SINK_PATH}")
print(f"📅 Evaluation Horizon        : {START_DATE} to {UNTIL_DATE}")
print(f"⏱️ Job Started At            : {start_dt_str}")
print("=" * 100)

pipeline = DatabricksReconciliationPipeline(
    spark_session=spark,
    database=DATABASE,
    s3_lakehouse_base=S3_LAKEHOUSE_BASE,
    s3_sink_path=S3_SINK_PATH,
    target_table=TARGET_TABLE,
    start_date=START_DATE,
    until_date=UNTIL_DATE,
    horizon_days=1,
    slot_minutes=5,
    critical_cover_hours=CRITICAL_THRESHOLD,
    monitor_cover_hours=MONITOR_THRESHOLD,
    write_mode=WRITE_MODE,
    dry_run=DRY_RUN
)

reconciliation_ledger_df = pipeline.run()
total_records = reconciliation_ledger_df.count()

job_end_time = time.time()
end_dt_str = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
total_duration_sec = job_end_time - job_start_time
mins, secs = divmod(total_duration_sec, 60)

dbus_consumed = (total_duration_sec / 3600.0) * DBU_RATE_PER_HOUR
estimated_cost_usd = dbus_consumed * DBU_DOLLAR_RATE


🚀 EXECUTING DATABRICKS RECONCILIATION ENGINE (2025-01-01 to 2026-09-10)
🏛️ AWS Glue Catalog Database : latest_elevate_data
☁️ AWS S3 Lakehouse Source   : s3://rakeshmohandas-uscentral1-825834484882-us-east-1-an/lakehouse/latest_elevate_data
☁️ AWS S3 Target Sink Bucket : s3://rakeshmohandas-uscentral1-825834484882-us-east-1-an/lakehouse/latest_elevate_data/gold_inventory_reconciliation_ledger
📅 Evaluation Horizon        : 2025-01-01 to 2026-09-10
⏱️ Job Started At            : 2026-08-30 17:41:23 UTC
🚀 Starting Databricks Multi-Day Inventory Reconciliation Pipeline | 2025-01-01 -> 2026-09-10
📍 AWS S3 Lakehouse Source Base : s3://rakeshmohandas-uscentral1-825834484882-us-east-1-an/lakehouse/latest_elevate_data
🏛️ AWS Glue Catalog Database    : latest_elevate_data
☁️ AWS S3 Target Sink           : s3://rakeshmohandas-uscentral1-825834484882-us-east-1-an/lakehouse/latest_elevate_data/gold_inventory_reconciliation_ledger
📋 Target Glue Table            : latest_elevate_data.gold_inventory_r

In [0]:
# Interactive Results Inspection & Risk Tier Categorization

# 1. Complete Conformed Reconciliation Ledger
print("📄 Complete Inventory Reconciliation Ledger:")
display(reconciliation_ledger_df)

# 2. Critical Stockout Incidents
print("\n🔴 Critical Stockout Incidents (Cover <= 6.0 hours):")
critical_df = reconciliation_ledger_df.filter(F.col("reconciliation_status") == STATUS_CRITICAL)
if critical_df.count() > 0:
    display(critical_df.select(
        "business_date", "store_id", "store_name", "city", "item_id", 
        "shelf_qty", "backroom_qty", "intraday_gross_revenue_usd",
        "est_cover_hours_remaining", "reconciliation_status"
    ))
else:
    print("✅ No critical stockouts detected in this evaluation window.")

# 3. Items Requiring Velocity Monitoring
print("\n🟡 Items Requiring Monitoring (6.0 < Cover <= 12.0 hours):")
monitor_df = reconciliation_ledger_df.filter(F.col("reconciliation_status") == STATUS_MONITOR)
if monitor_df.count() > 0:
    display(monitor_df.select(
        "business_date", "store_id", "store_name", "city", "item_id",
        "shelf_qty", "backroom_qty", "intraday_gross_revenue_usd",
        "est_cover_hours_remaining", "reconciliation_status"
    ))
else:
    print("✅ No items requiring velocity monitoring in this evaluation window.")

# 4. Status Distribution Summary
print("\n📊 Reconciliation Status Distribution Summary:")
status_summary_df = (reconciliation_ledger_df
                     .groupBy("reconciliation_status")
                     .agg(F.count("*").alias("record_count"),
                          F.round(F.sum("intraday_gross_revenue_usd"), 2).alias("total_revenue_usd"),
                          F.round(F.avg("est_cover_hours_remaining"), 1).alias("avg_cover_hours"))
                     .orderBy(F.col("record_count").desc()))
display(status_summary_df)

📄 Complete Inventory Reconciliation Ledger:


business_date,store_id,store_name,city,item_id,unit_price_usd,opening_qty,shelf_qty,backroom_qty,intraday_gross_revenue_usd,est_cover_hours_remaining,reconciliation_status
2026-09-01,STORE_012,Cymbal Madrid Gran Vía Flagship,Madrid,prod_8532,358.0,30,19,11,202473.4,4.7,CRITICAL BURN SPIKE - STOCKOUT RISK
2026-09-01,STORE_005,Cymbal Toronto Eaton Centre Galleria,Toronto,prod_8279,724.99,41,26,15,266292.79,5.3,CRITICAL BURN SPIKE - STOCKOUT RISK
2026-09-01,STORE_015,Cymbal Dubai Mall Grand Galleria,Dubai,prod_8279,724.99,89,57,32,384028.54,5.3,CRITICAL BURN SPIKE - STOCKOUT RISK
2026-09-01,STORE_016,Cymbal Sydney Harbour Waterfront Plaza,Sydney,prod_8279,724.99,92,59,33,283411.1,5.9,CRITICAL BURN SPIKE - STOCKOUT RISK
2026-09-01,STORE_009,Cymbal Berlin Kurfürstendamm Center,Berlin,prod_2194,999.99,99,64,35,362615.8,14.8,RECONCILED NORMAL HEALTH
2026-09-01,STORE_007,Cymbal Paris Champs-Élysées Flagship,Paris,prod_4691,1873.99,91,59,32,349161.28,15.2,RECONCILED NORMAL HEALTH
2026-09-01,STORE_003,Cymbal New York Fifth Avenue Megastore,New York,prod_8532,358.0,119,77,42,112126.85,15.4,RECONCILED NORMAL HEALTH
2026-09-01,STORE_015,Cymbal Dubai Mall Grand Galleria,Dubai,prod_9495,266.99,148,96,52,384028.54,17.6,RECONCILED NORMAL HEALTH
2026-09-01,STORE_002,Cymbal Los Angeles Century City Center,Los Angeles,prod_8279,724.99,134,87,47,264942.96,20.0,RECONCILED NORMAL HEALTH
2026-09-01,STORE_009,Cymbal Berlin Kurfürstendamm Center,Berlin,prod_8532,358.0,159,103,56,362615.8,36.7,RECONCILED NORMAL HEALTH



🔴 Critical Stockout Incidents (Cover <= 6.0 hours):


business_date,store_id,store_name,city,item_id,shelf_qty,backroom_qty,intraday_gross_revenue_usd,est_cover_hours_remaining,reconciliation_status
2026-09-01,STORE_012,Cymbal Madrid Gran Vía Flagship,Madrid,prod_8532,19,11,202473.4,4.7,CRITICAL BURN SPIKE - STOCKOUT RISK
2026-09-01,STORE_005,Cymbal Toronto Eaton Centre Galleria,Toronto,prod_8279,26,15,266292.79,5.3,CRITICAL BURN SPIKE - STOCKOUT RISK
2026-09-01,STORE_015,Cymbal Dubai Mall Grand Galleria,Dubai,prod_8279,57,32,384028.54,5.3,CRITICAL BURN SPIKE - STOCKOUT RISK
2026-09-01,STORE_016,Cymbal Sydney Harbour Waterfront Plaza,Sydney,prod_8279,59,33,283411.1,5.9,CRITICAL BURN SPIKE - STOCKOUT RISK
2026-07-01,STORE_012,Cymbal Madrid Gran Vía Flagship,Madrid,prod_8532,19,11,336775.71,4.7,CRITICAL BURN SPIKE - STOCKOUT RISK
2026-07-01,STORE_011,Cymbal Milan Via Montenapoleone Boutique,Milan,prod_4691,22,13,471399.84,5.0,CRITICAL BURN SPIKE - STOCKOUT RISK
2026-06-01,STORE_015,Cymbal Dubai Mall Grand Galleria,Dubai,prod_4691,29,17,473944.5,5.2,CRITICAL BURN SPIKE - STOCKOUT RISK
2026-06-01,STORE_007,Cymbal Paris Champs-Élysées Flagship,Paris,prod_2194,42,24,474557.7,5.2,CRITICAL BURN SPIKE - STOCKOUT RISK
2026-06-01,STORE_004,Cymbal Chicago Michigan Avenue Plaza,Chicago,prod_8279,26,15,231035.19,5.7,CRITICAL BURN SPIKE - STOCKOUT RISK
2026-05-01,STORE_015,Cymbal Dubai Mall Grand Galleria,Dubai,prod_8532,20,12,744951.66,4.8,CRITICAL BURN SPIKE - STOCKOUT RISK



🟡 Items Requiring Monitoring (6.0 < Cover <= 12.0 hours):


business_date,store_id,store_name,city,item_id,shelf_qty,backroom_qty,intraday_gross_revenue_usd,est_cover_hours_remaining,reconciliation_status
2026-08-01,STORE_013,Cymbal Tokyo Ginza District Flagship,Tokyo,prod_2194,39,22,287633.83,6.1,MONITOR VELOCITY
2026-08-01,STORE_002,Cymbal Los Angeles Century City Center,Los Angeles,prod_8532,37,21,183291.09,7.5,MONITOR VELOCITY
2026-07-01,STORE_015,Cymbal Dubai Mall Grand Galleria,Dubai,prod_8532,20,12,151515.89,6.3,MONITOR VELOCITY
2026-07-01,STORE_016,Cymbal Sydney Harbour Waterfront Plaza,Sydney,prod_8279,59,33,532177.9,9.5,MONITOR VELOCITY
2026-07-01,STORE_011,Cymbal Milan Via Montenapoleone Boutique,Milan,prod_9495,53,30,471399.84,11.5,MONITOR VELOCITY
2026-07-01,STORE_016,Cymbal Sydney Harbour Waterfront Plaza,Sydney,prod_4691,51,28,532177.9,11.9,MONITOR VELOCITY
2026-06-01,STORE_008,Cymbal San Francisco Union Square Flagship,San Francisco,prod_2194,22,12,222056.66,7.9,MONITOR VELOCITY
2026-05-01,STORE_013,Cymbal Tokyo Ginza District Flagship,Tokyo,prod_2194,39,22,255800.58,6.1,MONITOR VELOCITY
2026-05-01,STORE_003,Cymbal New York Fifth Avenue Megastore,New York,prod_4691,42,24,516834.77,6.2,MONITOR VELOCITY
2026-05-01,STORE_008,Cymbal San Francisco Union Square Flagship,San Francisco,prod_8279,26,14,467652.16,6.4,MONITOR VELOCITY



📊 Reconciliation Status Distribution Summary:


reconciliation_status,record_count,total_revenue_usd,avg_cover_hours
RECONCILED NORMAL HEALTH,8265,2.04036426234E9,767.7
MONITOR VELOCITY,84,3.297368742E7,7.7
CRITICAL BURN SPIKE - STOCKOUT RISK,51,2.430521174E7,5.1


In [0]:
# S3 Target Sink Verification & Read-Back Audit
print("=" * 100)
print("🔍 AUDITING AWS S3 SINK DATA INTEGRITY & PARITY")
print(f"📍 Reading from AWS S3 Target Sink: {S3_SINK_PATH}")
print("=" * 100)

try:
    # Read back directly from S3 Parquet / Delta Sink
    try:
        s3_audit_df = spark.read.parquet(S3_SINK_PATH)
    except Exception:
        s3_audit_df = spark.read.format("delta").load(S3_SINK_PATH)
    
    audit_count = s3_audit_df.count()
    
    print(f"✅ S3 Table Successfully Read directly from AWS S3 Lakehouse!")
    print(f"📊 Verified S3 Row Count : {audit_count:,} rows")
    print(f"📋 Verified S3 Columns   : {list(s3_audit_df.columns)}")
    
    # Assert exact match with pipeline output
    assert audit_count == total_records, f"Row count mismatch! S3 has {audit_count}, pipeline produced {total_records}"
    assert list(s3_audit_df.columns) == LEDGER_COLUMNS, f"Schema mismatch in S3 sink!"
    
    print("=" * 100)
    print("🎉 S3 AUDIT SUCCESS: 100% Data Integrity & Parity Confirmed in AWS S3 Lakehouse!")
    print("=" * 100)
except Exception as e:
    print(f"ℹ️ Audit notice: {e}")

🔍 AUDITING AWS S3 SINK DATA INTEGRITY & PARITY
📍 Reading from AWS S3 Target Sink: s3://rakeshmohandas-uscentral1-825834484882-us-east-1-an/lakehouse/latest_elevate_data/gold_inventory_reconciliation_ledger
✅ S3 Table Successfully Read directly from AWS S3 Lakehouse!
📊 Verified S3 Row Count : 8,400 rows
📋 Verified S3 Columns   : ['business_date', 'store_id', 'store_name', 'city', 'item_id', 'unit_price_usd', 'opening_qty', 'shelf_qty', 'backroom_qty', 'intraday_gross_revenue_usd', 'est_cover_hours_remaining', 'reconciliation_status']
🎉 S3 AUDIT SUCCESS: 100% Data Integrity & Parity Confirmed in AWS S3 Lakehouse!
